# Integração entre Dados das API's do IBGE e BC

Este notebook integrará os dados recebidos pelas API's do IBGE e do BC e irá uni-los em uma só tabela

## Imports

In [ ]:
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
import json

import altair as alt
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests as rqs
from IPython.display import display
from ipywidgets import Dropdown, HBox, Layout, VBox
from ipywidgets import Image as ImagemWidget
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.ticker import PercentFormatter


## Reunir os arquivos da API do BC

Os dados da API do BC estão na pasta dados > bcb > cache em formato tipo .json

In [ ]:
dados_da_api_bc = []

# Pesquisa todos os arquivos .json em dados > bcb > cache
for arquivo in Path("dados/bcb/cache").glob("*.json"):
    with open(arquivo, encoding="utf-8") as f:
        cache = json.load(f)
    dados_da_api_bc.append(pd.DataFrame(cache["dados"]["value"]))

sicor = pd.concat(dados_da_api_bc, ignore_index=True)
# Consultas sobrepostas podem repetir a mesma linha agregada do SICOR.
repetidos = sicor.duplicated().sum()
sicor = sicor.drop_duplicates(ignore_index=True)
print(f"Registros repetidos removidos do cache carregado: {repetidos:,}")


## Rodar o código da API_IBGE

Rodando o código com o %run e colocando o %%capture para não dar print em nada na tela, ou seja, apenas rodar e carregar todas as tabelas do API_IBGE

In [87]:
%%capture
%run API_IBGE.ipynb

## Padronizando os dados da API_BC

Padronizando os dados da API_BC para igualar os valores dos dados aos da API_IBGE e tratar a tabela

In [ ]:
# Renomeando as colunas com o mesmo nome das colunas da API_IBGE
sicor = sicor.rename(columns = {'nomeProduto':'cultura', 'AnoEmissao':'ano', 'MesEmissao':'mes', 'Municipio':'municipio' })

# Padronizando o nome da cultura
sicor['cultura'] = sicor['cultura'].replace({
    '"SOJA"':'Soja (em grão)',
    '"CANA-DE-AÇUCAR"' :'Cana-de-açúcar',
    '"MILHO"':'Milho (em grão)'
})

# Deixando o municipio em letra minúscula
sicor['municipio'] = sicor['municipio'].str.title()
sicor
# Renomenado as colunas relacionadas com Crédito rural
sicor = sicor.rename(columns={
    'cdFonteRecurso': 'fonte de recurso',
    'cdPrograma': 'programa de financiamento',
    'cdSubPrograma': 'subprograma',
    'cdModalidade': 'modalidade',
    'Atividade': 'atividade',
    'cdTipoSeguro': 'tipo de seguro',
    'VlCusteio': 'valor de custeio',
    'AreaCusteio': 'area de custeio',
})

#cria a tabela com os dados úteis do BC
tabela_bc = sicor.drop(columns=['codCadMu', 'cdProduto', 'codIbge','cdEstado'])

## Padronizando os dados da API_IBGE

A padronização será adicionar todos os dados municipais em uma única tabela

In [ ]:
#Unindo todas as tabelas com base nas colunas iguais
tabela_ibge_municipal = df_area_municipal.merge(df_prod_municipal, on=['cultura','codigo_municipio','ano','municipio','codigo_cultura'])
tabela_ibge_municipal = tabela_ibge_municipal.merge(df_rendimento_municipal, on=['cultura','codigo_municipio','ano','municipio','codigo_cultura'])

#Renomendo as colunas e retirando as colunas redundantes
tabela_ibge_municipal = tabela_ibge_municipal.drop(columns=['variavel_x','unidade_x','variavel_y','unidade_y', 'variavel','unidade'])
tabela_ibge_municipal = tabela_ibge_municipal.rename(columns={
    'valor_x':'area plantada (hectares)',
    'valor_y':'quantidade produzida (toneladas)',
    'valor':'rendimento medio (kg por hectare)',
    'codigo_cultura':'codigo cultura',
    'codigo_municipio':'codigo municipio'
})

# Adicionando a sigla do estado
tabela_ibge_municipal['estado'] = tabela_ibge_municipal['municipio'].str.rsplit(' - ', n=1).str[-1].str.strip()

# Padronizando o municipio para ficar idêntico ao da API_BC
tabela_ibge_municipal['municipio'] = tabela_ibge_municipal['municipio'].str.rsplit(" - ", n=1).str[0].str.strip().str.title().astype(str)

# Ordenar
nova_ordem = [
    'ano',
    'estado',
    'municipio',
    'codigo municipio',
    'cultura',
    'codigo cultura',
    'area plantada (hectares)',
    'quantidade produzida (toneladas)',
    'rendimento medio (kg por hectare)'
]
tabela_ibge_municipal = tabela_ibge_municipal[nova_ordem]
tabela_ibge_municipal

## Dicionário para padronização da 2° tabela (a seguir)

In [90]:
programas = {
    "0001": "PRONAF",
    "0050": "PRONAMP",
    "0999": "Sem vínculo a programa específico",
    "0070": "Funcafé",
    "0151": "Procap-Agro",
    "0152": "Proirriga (antigo Moderinfra)",
    "0153": "Moderagro",
    "0154": "Moderfrota",
    "0155": "Prodecoop",
    "0162": "Inovagro",
    "0163": "PCA (armazéns)",
    "0201": "Crédito Fundiário (FTRA)",
    "0222": "Renovagro",
    "0240": "ANF (atividade não financiada, Proagro)",
}

fontes = {
    "0100": "TESOURO NACIONAL - DIRECIONADA/CONTROLADA",
    "0201": "OBRIGATÓRIOS - MCR 6.2 - DIRECIONADA/CONTROLADA",
    "0202": "EXIGIBILIDADE ADICIONAL DOS RECURSOS À VISTA",
    "0222": "Exigibilidade Adicional dos Recursos à Vista - Resolução 5030 ENCERRADO",
    "0225": "Exigibilidade Adicional dos Recursos à Vista - Resolução 5087 - ENCERRADO",
    "0226": "Exigibilidade Adicional dos Recursos à Vista - Resolução 5157",
    "0250": "FACULDADE DE APLICAÇÃO - COMPULSÓRIO",
    "0260": "COMPULSÓRIO SOBRE RECURSOS À VISTA - REFORÇO DO INVESTIMENTO (CIRC 3.745)",
    "0300": "POUPANÇA RURAL - EQUALIZADA - DIRECIONADA/CONTROLADA",
    "0301": "POUPANÇA RURAL - CONTROLADOS - CONDIÇÕES MCR 6.2",
    "0302": "POUPANÇA RURAL - CONTROLADOS - FATOR DE PONDERAÇÃO",
    "0303": "POUPANÇA RURAL - DIRECIONADA/NÃO CONTROLADA",
    "0304": "EXIGIBILIDADE ADICIONAL DA POUPANÇA RURAL",
    "0402": "RECURSOS LIVRES - LIVRE/NÃO CONTROLADA",
    "0403": "RECURSOS LIVRES - EQUALIZADA - LIVRE/CONTROLADA",
    "0405": "FUNDO DE COMMODITIES",
    "0430": "LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - DIRECIONADA/NÃO CONTROLADA",
    "0431": "LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - EQUALIZADA - DIRECIONADA/CONTROLADA",
    "0440": "LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - TAXA FAVORECIDA",
    "0450": "INSTR HIBRIDO CAPITAL DÍVIDA-IHCD (Lei 12.793/2013 - Art. 6º) - EQUALIZÁVEL",
    "0451": "INSTR HIBRIDO CAPITAL DÍVIDA-IHCD (Lei 12.793/2013 - Art. 6º) - DIRECIONADA",
    "0501": "FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO NORTE (FNO) - DIRECIONADA/CONTROLADA",
    "0502": "FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO NORDESTE (FNE) - DIRECIONADA/CONTROLADA",
    "0503": "FUNDO CONSTITUCIONAL DE FINANCIAMENTO DO CENTRO-OESTE (FCO) - DIRECIONADA/CONTROLADA",
    "0505": "BNDES/FINAME - EQUALIZADA - DIRECIONADA/CONTROLADA",
    "0506": "BNDES - LIVRE/NÃO CONTROLADA",
    "0507": "INCRA - DIRECIONADA/CONTROLADA",
    "0520": "FUNDO DE TERRAS E DA REFORMA AGRÁRIA - DIRECIONADA/CONTROLADA",
    "0600": "GOVERNOS E FUNDOS ESTADUAIS OU MUNICIPAIS - DIRECIONADA/CONTROLADA",
    "0650": "FAT - FUNDO DE AMPARO AO TRABALHADOR - DIRECIONADA/CONTROLADA",
    "0680": "PIS/PASEP",
    "0800": "FUNCAFE - FUNDO DE DEFESA DA ECONOMIA CAFEEIRA - DIRECIONADA/CONTROLADA",
    "0850": "CAPTAÇÃO EXTERNA - LIVRE/NÃO CONTROLADA",
    "0900": "ATIVIDADE NÃO FINANCIADA ENQUADRADA NO PROAGRO (MCR 16.8)",
    "0910": "FUNDO NACIONAL SOBRE MUDANÇA (FUNDO CLIMA) - RES CMN 5.130/2024",
    "0911": "FUNDO DE DESENVOLVIMENTO CIENTÍFICO E TECNOLÓGICO (FNDCT) - MP 1374 DIRECIONADA/CONTROLADA",
    "0990": "OUTRAS FONTES DE RECURSOS NÃO ESPECIFICADAS",
}

tipo_seguro = {
    0: "Não se aplica",
    1: "Proagro tradicional",
    2: "Proagro mais",
    3: "Outro seguro",
    9: "Sem adesão a seguro",
}

meses = {
    "01": "Janeiro",
    "02": "Fevereiro",
    "03": "Março",
    "04": "Abril",
    "05": "Maio",
    "06": "Junho",
    "07": "Julho",
    "08": "Agosto",
    "09": "Setembro",
    "10": "Outubro",
    "11": "Novembro",
    "12": "Dezembro"
}


## Criação de duas tabelas com os dados do BC

1° Tabela: tabela agrupada com o valor do crédito rural por municipio, cultura e ano

2° Tabela: tabela padronizada com os nomes de cada coluna corretamente

In [91]:
# 1° Tabela: tabela do crédito rural
colunas_a_serem_somadas = ['valor de custeio', 'area de custeio']
tabela_credito_rural = sicor.groupby(['municipio','cultura','ano'])[colunas_a_serem_somadas].sum().reset_index()
# Transformando o ano em int
tabela_credito_rural['ano'] = tabela_credito_rural['ano'].astype(int)

# 2° Tabela: tabela com os nomes padronizados com o dicionário de cima
tabela_padronizada = sicor.copy()
tabela_padronizada['programa de financiamento'] = sicor['programa de financiamento'].replace(programas)
tabela_padronizada['fonte de recurso'] = sicor['fonte de recurso'].replace(fontes)
tabela_padronizada['tipo de seguro'] = sicor['tipo de seguro'].replace(tipo_seguro)
# ANALISAR SE PRECISA PADRONIZAR O MÊS
# tabela_padronizada['mes'] = sicor['mes'].replace(meses)
tabela_padronizada['mes'] = sicor['mes'].astype(int)

tabela_padronizada


,municipio,cultura,mes,ano,programa de financiamento,subprograma,fonte de recurso,tipo de seguro,cdEstado,valor de custeio,cdProduto,codCadMu,atividade,modalidade,codIbge,area de custeio
0,Jesuítas,Soja (em grão),7,2021,Sem vínculo a programa específico,0,POUPANÇA RURAL - DIRECIONADA/NÃO CONTROLADA,9,19,120515.25,6720,40785,1,01,4112751,40.29
1,Flor Da Serra Do Sul,Soja (em grão),7,2021,PRONAF,1,POUPANÇA RURAL - EQUALIZADA - DIRECIONADA/CONT...,3,19,884242.51,6720,49982,1,01,4107850,264.96
2,Dianópolis,Soja (em grão),7,2021,Sem vínculo a programa específico,0,LETRA DE CRÉDITO DO AGRONEGÓCIO (LCA) - DIRECI...,3,28,354260.00,6720,12036,1,01,1707009,100.00
3,Nova Araçá,Soja (em grão),7,2021,PRONAMP,0,BNDES/FINAME - EQUALIZADA - DIRECIONADA/CONTRO...,1,24,44311.34,6720,34832,1,01,4312807,12.00
4,Iporã,Soja (em grão),7,2021,PRONAMP,0,BNDES/FINAME - EQUALIZADA - DIRECIONADA/CONTRO...,3,19,129270.72,6720,30953,1,01,4110607,42.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
831985,Caseara,Soja (em grão),1,2018,Sem vínculo a programa específico,0,POUPANÇA RURAL - DIRECIONADA/NÃO CONTROLADA,9,28,1150000.00,6720,43751,1,01,1703909,564.89
831986,Presidente Bernardes,Soja (em grão),1,2018,PRONAF,1,POUPANÇA RURAL - EQUALIZADA - DIRECIONADA/CONT...,2,27,27267.81,6720,40163,1,01,3541208,16.94
831987,Nobres,Soja (em grão),1,2018,Sem vínculo a programa específico,0,OBRIGATÓRIOS - MCR 6.2 - DIRECIONADA/CONTROLADA,3,14,2606343.93,6720,12988,1,01,5105903,1278.70
831988,Águas Frias,Soja (em grão),1,2018,PRONAF,1,POUPANÇA RURAL - EQUALIZADA - DIRECIONADA/CONT...,2,25,5892.00,6720,49494,1,01,4200556,4.00


## Unir as tabelas

Irá unir-se as tabelas: tabela_ibge_municipal com tabela_credito_rural. Entretanto, há alguns valores que estão apenas em uma das duas tabelas, logo esses valores possuirão NaN e serão removidos.

In [92]:
tabela_unida = tabela_ibge_municipal.merge(tabela_credito_rural, on=['municipio', 'ano','cultura'])
tabela_unida = tabela_unida.dropna()
tabela_unida

,ano,estado,municipio,codigo municipio,cultura,codigo cultura,area plantada (hectares),quantidade produzida (toneladas),rendimento medio (kg por hectare),valor de custeio,area de custeio
0,2021,GO,Abadia De Goiás,5200050,Milho (em grão),2711,200.0,1360.0,6800.0,1220142.65,463.97
1,2023,GO,Abadia De Goiás,5200050,Milho (em grão),2711,300.0,1332.0,4440.0,1365264.35,394.46
2,2016,GO,Abadia De Goiás,5200050,Soja (em grão),2713,0.0,0.0,0.0,293962.23,160.55
3,2017,GO,Abadia De Goiás,5200050,Soja (em grão),2713,217.0,720.0,3318.0,399997.72,209.39
4,2018,GO,Abadia De Goiás,5200050,Soja (em grão),2713,197.0,621.0,3152.0,405726.63,209.39
...,...,...,...,...,...,...,...,...,...,...,...
49099,2021,SP,Óleo,3533809,Soja (em grão),2713,5890.0,24792.0,4209.0,3529324.92,1043.23
49100,2022,SP,Óleo,3533809,Soja (em grão),2713,5890.0,24765.0,4205.0,7584973.10,1292.92
49101,2023,SP,Óleo,3533809,Soja (em grão),2713,5143.0,20402.0,3967.0,8117759.43,1607.08
49102,2024,SP,Óleo,3533809,Soja (em grão),2713,5143.0,10243.0,1992.0,6289436.44,1176.02


## Base validada

Prepara os dados das perguntas 4 e 5, mantendo cobertura nas duas fontes, área plantada positiva e crédito numérico, finito e não negativo. A função informa as exclusões e salva a tabela validada sem substituir a base completa; ausência de crédito ou contratos não é convertida em zero. As conclusões representam apenas os municípios cobertos pelas fontes.


In [93]:
def validar_tabela(tabela):
    """Filtra cobertura ambos, area positiva e credito finito nao negativo."""
    validada = tabela.copy()
    total = len(validada)

    anterior = len(validada)
    validada = validada.loc[validada["cobertura"].eq("ambos")].copy()
    print(f"Excluidos por cobertura: {anterior - len(validada):,}")

    area = pd.to_numeric(validada["area_plantada_ha"], errors="coerce")
    anterior = len(validada)
    validada["area_plantada_ha"] = area
    validada = validada.loc[(area.gt(0) & area.lt(float("inf"))).fillna(False)].copy()
    print(f"Excluidos por area ausente, invalida ou nao positiva: {anterior - len(validada):,}")

    credito = pd.to_numeric(validada["credito_nominal_brl"], errors="coerce")
    anterior = len(validada)
    validada["credito_nominal_brl"] = credito
    validada = validada.loc[(credito.ge(0) & credito.lt(float("inf"))).fillna(False)].copy()
    print(f"Excluidos por credito ausente, invalido ou negativo: {anterior - len(validada):,}")

    if "quantidade_contratos" not in validada.columns:
        validada["quantidade_contratos"] = pd.Series(pd.NA, index=validada.index, dtype="Int64")
    print(f"Total: {total:,} | Excluidos: {total - len(validada):,} | Mantidos: {len(validada):,}")
    print(f"Registros sem quantidade de contratos: {validada['quantidade_contratos'].isna().sum():,}")
    return validada.reset_index(drop=True)


tabela_completa = pd.read_csv("dados/tabela_final.csv", dtype={"codigo_municipio": "string"})
tabela_validada = validar_tabela(tabela_completa)
tabela_validada.to_csv("dados/tabela_validada.csv", index=False)
tabela_validada.head()


Excluidos por cobertura: 110,788
Excluidos por area ausente, invalida ou nao positiva: 1,335
Excluidos por credito ausente, invalido ou negativo: 0
Total: 164,775 | Excluidos: 112,123 | Mantidos: 52,652
Registros sem quantidade de contratos: 52,652


,codigo_municipio,municipio,uf,regiao,codigo_uf_ibge,cultura,ano,cobertura,area_plantada_ha,producao_t,rendimento_kg_ha,credito_nominal_brl,area_financiada_ha,quantidade_contratos
0,1100015,Alta Floresta D'Oeste - RO,RO,Norte,11,milho,2018,ambos,6728.0,16267.0,2418.0,700000.00,671.00,<NA>
1,1100015,Alta Floresta D'Oeste - RO,RO,Norte,11,milho,2019,ambos,6728.0,16267.0,2418.0,652809.80,600.00,<NA>
2,1100015,Alta Floresta D'Oeste - RO,RO,Norte,11,milho,2020,ambos,6728.0,16267.0,2418.0,1016508.71,550.31,<NA>
3,1100015,Alta Floresta D'Oeste - RO,RO,Norte,11,milho,2021,ambos,10562.0,31550.0,2987.0,716085.34,248.89,<NA>
4,1100015,Alta Floresta D'Oeste - RO,RO,Norte,11,milho,2022,ambos,1344.0,5556.0,4134.0,1552666.34,491.02,<NA>


## Faixas de área plantada

Apoia a pergunta 4 agrupando áreas próximas dentro de cada cultura e ano: os limites começam na menor área observada e crescem 25% por faixa, com amplitude ajustável. O limite inferior é incluído e o superior pertence à faixa seguinte; os rótulos são arredondados, mas os limites numéricos são preservados. Compare as faixas pelos hectares, não apenas pelo número do grupo, e observe quantos municípios participam de cada uma.


In [ ]:
def categorizar_faixas_hectares(tabela, amplitude=0.25):
    """Categoriza a tabela validada em faixas relativas por cultura e ano."""
    fator = 1 + amplitude
    if not 1 < fator < float("inf"):
        raise ValueError("A amplitude deve ser finita e maior que zero.")

    partes = []
    for _, grupo in tabela.groupby(["cultura", "ano"], sort=False):
        grupo = grupo.copy()
        area = grupo["area_plantada_ha"]
        limites = [float(area.min())]
        while limites[-1] <= area.max():
            limites.append(limites[-1] * fator)

        faixa = pd.cut(area, bins=limites, labels=False, right=False)
        grupo["faixa_area_ordem"] = faixa + 1
        grupo["area_min_faixa_ha"] = faixa.map(dict(enumerate(limites[:-1])))
        grupo["area_max_faixa_ha"] = faixa.map(dict(enumerate(limites[1:])))
        grupo["faixa_hectares"] = (
            "[" + grupo["area_min_faixa_ha"].round(2).astype(str)
            + "; " + grupo["area_max_faixa_ha"].round(2).astype(str) + ") ha"
        )
        partes.append(grupo)

    if not partes:
        return tabela.assign(
            faixa_area_ordem=pd.Series(dtype="int64"),
            area_min_faixa_ha=pd.Series(dtype="float64"),
            area_max_faixa_ha=pd.Series(dtype="float64"),
            faixa_hectares=pd.Series(dtype="string"),
        )
    return pd.concat(partes).sort_index()


tabela_categorizada = categorizar_faixas_hectares(tabela_validada)
try:
    tabela_categorizada.to_csv("dados/tabela_categorizada.csv", index=False)
except OSError as erro:
    print(f"CSV não atualizado: {erro}. A tabela permanece disponível em memória para os gráficos.")
resumo_faixas = (
    tabela_categorizada.groupby(
        ["cultura", "ano", "faixa_area_ordem", "faixa_hectares"], as_index=False
    ).agg(municipios=("codigo_municipio", "nunique"))
)
print(f"Registros categorizados: {len(tabela_categorizada):,}")
print(f"Faixas ocupadas por cultura e ano: {len(resumo_faixas):,}")
print(f"Faixas com apenas um municipio: {resumo_faixas['municipios'].eq(1).sum():,}")
tabela_categorizada.head()


## Estatísticas por faixa

Apoia a pergunta 4 com número de municípios, mediana da área e mediana, quartis e intervalo interquartil do crédito e do crédito por hectare, calculado primeiro para cada município. Caixas ou intervalos mais amplos indicam maior dispersão, mas grupos com apenas um município não permitem comparar variações internas. Estatísticas de contratos são incluídas somente quando existem contagens válidas; a tabela atual não fornece essa informação.


In [ ]:
def primeiro_quartil(serie):
    return serie.quantile(0.25)


def terceiro_quartil(serie):
    return serie.quantile(0.75)


def resumir_estatisticas_faixas(tabela):
    """Resume area, credito e, quando disponiveis, contratos por faixa."""
    base = tabela.copy()
    chaves = ["cultura", "ano", "faixa_area_ordem", "faixa_hectares"]
    base["credito_por_hectare_brl"] = base["credito_nominal_brl"] / base["area_plantada_ha"]
    estatisticas = {
        "area_min_faixa_ha": ("area_min_faixa_ha", "first"),
        "area_max_faixa_ha": ("area_max_faixa_ha", "first"),
        "numero_municipios": ("codigo_municipio", "nunique"),
        "mediana_area_ha": ("area_plantada_ha", "median"),
        "mediana_credito_brl": ("credito_nominal_brl", "median"),
        "q1_credito_brl": ("credito_nominal_brl", primeiro_quartil),
        "q3_credito_brl": ("credito_nominal_brl", terceiro_quartil),
        "mediana_credito_por_hectare_brl": ("credito_por_hectare_brl", "median"),
        "q1_credito_por_hectare_brl": ("credito_por_hectare_brl", primeiro_quartil),
        "q3_credito_por_hectare_brl": ("credito_por_hectare_brl", terceiro_quartil),
    }

    contratos_disponiveis = False
    if "quantidade_contratos" in base.columns:
        contratos = pd.to_numeric(base["quantidade_contratos"], errors="coerce")
        base["quantidade_contratos"] = contratos.where(
            contratos.ge(0) & contratos.lt(float("inf")) & contratos.mod(1).eq(0)
        )
        estatisticas["municipios_com_contratos"] = ("quantidade_contratos", "count")
        contratos_disponiveis = base["quantidade_contratos"].notna().any()
        if contratos_disponiveis:
            base["contratos_por_mil_hectares"] = base["quantidade_contratos"] * 1000 / base["area_plantada_ha"]
            base["valor_medio_por_contrato_brl"] = base["credito_nominal_brl"] / base["quantidade_contratos"].where(base["quantidade_contratos"].gt(0))
            estatisticas.update({
                "mediana_contratos": ("quantidade_contratos", "median"),
                "q1_contratos": ("quantidade_contratos", primeiro_quartil),
                "q3_contratos": ("quantidade_contratos", terceiro_quartil),
                "mediana_contratos_por_mil_hectares": ("contratos_por_mil_hectares", "median"),
                "q1_contratos_por_mil_hectares": ("contratos_por_mil_hectares", primeiro_quartil),
                "q3_contratos_por_mil_hectares": ("contratos_por_mil_hectares", terceiro_quartil),
                "mediana_valor_medio_por_contrato_brl": ("valor_medio_por_contrato_brl", "median"),
                "municipios_com_valor_por_contrato": ("valor_medio_por_contrato_brl", "count"),
            })

    resumo = base.groupby(chaves, as_index=False).agg(**estatisticas)
    resumo["iqr_credito_brl"] = resumo["q3_credito_brl"] - resumo["q1_credito_brl"]
    resumo["iqr_credito_por_hectare_brl"] = resumo["q3_credito_por_hectare_brl"] - resumo["q1_credito_por_hectare_brl"]
    if contratos_disponiveis:
        resumo["iqr_contratos"] = resumo["q3_contratos"] - resumo["q1_contratos"]
        resumo["iqr_contratos_por_mil_hectares"] = resumo["q3_contratos_por_mil_hectares"] - resumo["q1_contratos_por_mil_hectares"]
    return resumo.sort_values(["cultura", "ano", "faixa_area_ordem"]).reset_index(drop=True)


tabela_estatistica = resumir_estatisticas_faixas(tabela_categorizada)
tabela_estatistica.to_csv("dados/tabela_estatistica_faixas.csv", index=False)
print(f"Grupos resumidos: {len(tabela_estatistica):,}")
print(f"Grupos com apenas um municipio: {tabela_estatistica['numero_municipios'].eq(1).sum():,}")
if "mediana_contratos" not in tabela_estatistica.columns:
    print("Sem contagens validas de contratos na base atual; estatisticas de contratos nao calculadas.")
display(tabela_estatistica.head(10))


## Gráficos

### Variáveis compartilhadas

Centraliza as regiões, as chaves de cultura e ano e as cores reutilizadas pelos gráficos. Os limites de área e de municípios preservam os critérios dos gráficos 6 e 7; opções de seletores de bases diferentes continuam sendo calculadas a partir dos respectivos dados.


In [ ]:
REGIOES = {"1": "Norte", "2": "Nordeste", "3": "Sudeste", "4": "Sul", "5": "Centro-Oeste"}

CHAVES_CULTURA_ANO = ["cultura", "ano"]
# Paleta compartilhada: azul para crédito e verde para indicadores agrícolas.
AZUL = "#2866A6"
VERDE = "#3A851D"
VERDE_AZULADO = "#176B65"
VERMELHO = "#B64040"
CINZA_FUNDO = "#EDF0F2"
CINZA_TIPICOS = "#B8BEC4"
CINZA_CONTORNO = "#65717C"
CINZA_TEXTO = "#263238"

CORES_CREDITO_AGRICULTURA = [AZUL, VERDE]
TONS_AZUL = ["#D6E4F2", "#9BC1E3", "#5B97CF", AZUL]
TONS_VERDE = ["#D5E8CF", "#A8D19A", "#6FB05B", VERDE]

CORES_CULTURAS = alt.Scale(
    domain=['Soja (em grão)', 'Milho (em grão)', 'Cana-de-açúcar'],
    range=[VERDE, VERMELHO, VERDE_AZULADO],
)
# Ordem visual aproximada de norte a sul, sem alterar os códigos do IBGE.
ORDEM_REGIOES = ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]
CORES_REGIOES = alt.Scale(
    domain=ORDEM_REGIOES,
    range=[VERDE, "#359044", "#2C9576", "#2B8296", AZUL],
)
CORES_ATIPICOS = {
    "Crédito/ha atípico": AZUL,
    "Produtividade atípica": VERDE,
    "Ambos": VERMELHO,
}

AREA_MINIMA_HA = 50
MIN_MUNICIPIOS_UF = 10


### Gráfico 1 — Evolução do crédito e dos contratos por cultura

Apoia a pergunta 1 mostrando a evolução mensal do valor de custeio e uma segunda série de contagens por cultura: linhas ascendentes indicam crescimento e as diferenças entre cores mostram trajetórias distintas. Os valores monetários são nominais; no código atual, a segunda série conta registros da tabela, portanto não deve ser interpretada como número de contratos sem validar a unidade de cada registro e a ausência de duplicações na fonte.


In [ ]:
def criar_grafico_evolucao_credito():
    tabela_valor_contratado = sicor.groupby(['cultura', 'ano','mes'])['valor de custeio'].sum().reset_index()

    # Coluna com a data formatada adequadamente
    tabela_valor_contratado['data'] = pd.to_datetime({
        'year': tabela_valor_contratado['ano'],
        'month': tabela_valor_contratado['mes'],
        # o dia é padronizado como o 1° dia de cada mês, porque a função exige que se passe um dia
        'day': 1
    })


    grafico_valor_contratado = alt.Chart(tabela_valor_contratado).mark_line().encode(
        x=alt.X('yearmonth(data):T', 
            title='Mês / Ano',
            axis=alt.Axis(format="%m/%Y", tickCount={"interval": "month", "step": 6})
        ),
        y=alt.Y('valor de custeio:Q',
            title='Valor de Custeio'
        ),
        color=alt.Color('cultura:N', scale=CORES_CULTURAS, title= 'Culturas')
    ).properties(
        title='Evolução do Valor de Custeio por Cultura',
        width=700,
        height=400
    )
    return grafico_valor_contratado


In [ ]:
def criar_grafico_evolucao_registros():
    tabela_numero_contratos = sicor.groupby(['cultura','ano','mes']).size().reset_index(name='quantidade')

    # Coluna com a data formatada adequadamente
    tabela_numero_contratos['data'] = pd.to_datetime({
        'year': tabela_numero_contratos['ano'],
        'month': tabela_numero_contratos['mes'],
        # o dia é padronizado como o 1° dia de cada mês, porque a função exige que se passe um dia
        'day': 1
    })


    grafico_numero_contratos = alt.Chart(tabela_numero_contratos).mark_line().encode(
        x=alt.X('yearmonth(data):T', 
            title='Mês / Ano',
            axis=alt.Axis(format="%m/%Y", tickCount={"interval": "month", "step": 6})
        ),
        y=alt.Y('quantidade:Q',
            title='Quantidade de Contratos',
        ),
        color=alt.Color('cultura:N', scale=CORES_CULTURAS, title= 'Culturas')
    ).properties(
        title='Evolução da Quantidade de Contratos por Cultura',
        width=700,
        height=400
    )
    return grafico_numero_contratos


def exibir_grafico_1():
    with alt.renderers.enable("default", embed_options={"actions": False}):
        display(criar_grafico_evolucao_credito())
        display(criar_grafico_evolucao_registros())


### Gráfico 2 — Distribuição do crédito e da área entre estados e regiões

Responde à pergunta 2 comparando a participação do crédito e da área plantada no total nacional de cada cultura, nos anos comuns às fontes. Escolha a cultura e compare as barras: participação do crédito superior à da área indica maior peso relativo no financiamento, não maior valor absoluto por hectare. Os oito estados com maior área aparecem separados e os restantes formam “Outros”; os resultados descrevem o período agregado.


In [ ]:
# Comparar crédito nominal e área plantada nos anos em comum de cada cultura.
area_g2 = df_area_estadual.rename(columns={"codigo_estado": "uf", "valor": "Área plantada (ha)"})
area_g2 = area_g2.assign(uf=area_g2["uf"].astype(str), ano=area_g2["ano"].astype(int))
credito_g2 = sicor.assign(
    uf=sicor["codIbge"].astype(str).str[:2],
    ano=sicor["ano"].astype(int),
).rename(columns={"valor de custeio": "Crédito de custeio (R$)"})

# Usar apenas os anos que aparecem nas duas fontes para cada cultura.
anos_comuns = area_g2[CHAVES_CULTURA_ANO].drop_duplicates().merge(
    credito_g2[CHAVES_CULTURA_ANO].drop_duplicates(), on=CHAVES_CULTURA_ANO
)
periodos = anos_comuns.groupby("cultura")["ano"].agg(["min", "max"])

indicadores_g2 = ["Crédito de custeio (R$)", "Área plantada (ha)"]
chave_uf = ["cultura", "uf"]
credito_comum = credito_g2.merge(anos_comuns, on=CHAVES_CULTURA_ANO)
area_comum = area_g2.merge(anos_comuns, on=CHAVES_CULTURA_ANO)

# Somar cada indicador por UF e juntar os resultados, incluindo UFs com dados em uma só fonte.
credito_por_uf = credito_comum.groupby(chave_uf)[indicadores_g2[0]].sum()
area_por_uf = area_comum.groupby(chave_uf)[indicadores_g2[1]].sum()
totais_g2 = pd.concat([credito_por_uf, area_por_uf], axis=1).fillna(0).reset_index()

nomes_uf = area_g2[["uf", "estado"]].drop_duplicates().set_index("uf")["estado"]
totais_g2["estado"] = totais_g2["uf"].map(nomes_uf).fillna("UF não identificada")
totais_g2["regiao"] = totais_g2["uf"].str[0].map(REGIOES).fillna("Região não identificada")

limite_estados = 8


def grafico_2(cultura):
    dados = totais_g2[totais_g2["cultura"] == cultura]
    por_regiao = dados.groupby("regiao")[indicadores_g2].sum()
    por_estado = dados.groupby("estado")[indicadores_g2].sum()

    # Ordenar pela área plantada e somar o restante em uma única categoria.
    por_estado = por_estado.sort_values("Área plantada (ha)", ascending=False, kind="stable")
    principais = por_estado.head(limite_estados).copy()
    demais = por_estado.iloc[limite_estados:]
    if not demais.empty:
        principais.loc["Outros"] = demais.sum()

    # Dividir cada indicador pela soma do Brasil.
    total_brasil = por_estado.sum().replace(0, float("nan"))
    participacao_regiao = por_regiao / total_brasil
    participacao_estado = principais / total_brasil


    with plt.ioff():
        fig, (ax_regiao, ax_estado) = plt.subplots(2, 1, figsize=(13.5, 7), layout="constrained")
        fig.get_layout_engine().set(hspace=0.25)
        participacao_regiao.plot.barh(ax=ax_regiao, color=CORES_CREDITO_AGRICULTURA, legend=False, width=0.8)
        participacao_estado.plot.bar(ax=ax_estado, color=CORES_CREDITO_AGRICULTURA, legend=False, rot=30, width=0.8)

        ax_regiao.set(title="Região", xlabel="Participação no total nacional", ylabel="")
        ax_regiao.title.set_fontsize(16)
        ax_estado.set(title="Estados", xlabel="", ylabel="Participação no total nacional")
        ax_estado.title.set_fontsize(16)
        ax_regiao.xaxis.set_major_formatter(PercentFormatter(1))
        ax_estado.yaxis.set_major_formatter(PercentFormatter(1))
        ax_regiao.legend(
            loc="lower center", bbox_to_anchor=(0.5, 1.18), ncol=2, frameon=False,
        )

        for eixo, direcao in [(ax_regiao, "x"), (ax_estado, "y")]:
            eixo.set_axisbelow(True)
            eixo.grid(axis=direcao, alpha=0.22)
            eixo.spines[["top", "right", "left"]].set_visible(False)
        for rotulo in ax_estado.get_xticklabels():
            rotulo.set_ha("right")

        primeiro_ano, ultimo_ano = periodos.loc[cultura]
        fig.suptitle(
            f"Distribuição do crédito de custeio e da área plantada\n"
            f"{cultura} · {primeiro_ano} a {ultimo_ano}",
            fontsize=18,
        )
        ax_regiao.tick_params(axis="y", labelsize=14)
        ax_regiao.tick_params(axis="x", labelsize=12)
        ax_estado.tick_params(axis="both", labelsize=12)
        buffer = BytesIO()
        fig.savefig(buffer, format="png", dpi=100, bbox_inches="tight")
        imagem = buffer.getvalue()
        buffer.close()
        plt.close(fig)

    return imagem


def exibir_grafico_2():
    seletor_g2 = Dropdown(
        options=periodos.index.tolist(), description="Cultura", layout=Layout(width="330px"),
    )
    imagem_g2 = ImagemWidget(format="png", layout=Layout(max_width="100%"))


    def atualizar_imagem_g2(mudanca=None):
        imagem_g2.value = grafico_2(seletor_g2.value) or b""


    seletor_g2.observe(atualizar_imagem_g2, names="value")
    atualizar_imagem_g2()
    display(VBox([seletor_g2, imagem_g2]))


### Gráfico 3 — Concentração municipal do crédito e da produção

Responde à pergunta 3 comparando quanto do crédito e da produção fica nos municípios que lideram cada ranking, para a cultura selecionada e o período indicado. Uma participação maior nos grupos de topo indica maior concentração; os rankings dos dois indicadores podem conter municípios diferentes. Top 1%, 5% e 10% são grupos sobrepostos e não devem ser somados; a comparação também depende do tratamento das ausências, que o código atual preenche com zero.


In [ ]:
credito_g3 = sicor[["cultura", "ano", "codIbge", "valor de custeio"]].rename(
    columns={"codIbge": "municipio_id", "valor de custeio": "Crédito de custeio (R$)"}
)
producao_g3 = df_prod_municipal[["cultura", "ano", "codigo_municipio", "valor"]].rename(
    columns={"codigo_municipio": "municipio_id", "valor": "Produção (t)"}
)
credito_g3["municipio_id"] = credito_g3["municipio_id"].astype(str)
producao_g3["municipio_id"] = producao_g3["municipio_id"].astype(str)
credito_g3["ano"] = credito_g3["ano"].astype(int)
producao_g3["ano"] = producao_g3["ano"].astype(int)

# Comparar os dois indicadores nos mesmos anos para cada cultura.
anos_g3 = credito_g3[CHAVES_CULTURA_ANO].drop_duplicates().merge(
    producao_g3[CHAVES_CULTURA_ANO].drop_duplicates(), on=CHAVES_CULTURA_ANO
)
periodos_g3 = anos_g3.groupby("cultura")["ano"].agg(["min", "max"])

chave_municipio_g3 = ["cultura", "municipio_id"]
coluna_credito = "Crédito de custeio (R$)"
coluna_producao = "Produção (t)"
credito_municipal = (
    credito_g3.merge(anos_g3, on=CHAVES_CULTURA_ANO)
    .groupby(chave_municipio_g3)[coluna_credito].sum()
)
producao_municipal = (
    producao_g3.merge(anos_g3, on=CHAVES_CULTURA_ANO)
    .groupby(chave_municipio_g3)[coluna_producao].sum()
)
# Incluir municípios com dados em uma só fonte, usando zero no outro indicador.
totais_g3 = pd.concat([credito_municipal, producao_municipal], axis=1).fillna(0).reset_index()


def grafico_3(cultura):
    dados = totais_g3[totais_g3["cultura"] == cultura]
    faixas = [1, 5, 10]
    rotulos = [f"Top {percentual}%" for percentual in faixas] + ["Demais municípios"]
    resumo = pd.DataFrame(index=rotulos)

    for indicador in [coluna_credito, coluna_producao]:
        valores = dados[indicador].sort_values(ascending=False)
        total = valores.sum()
        participacoes = []
        for percentual in faixas:
            # Arredondar para cima para incluir pelo menos um município.
            quantidade = (len(valores) * percentual + 99) // 100
            parcela = valores.head(quantidade).sum()
            participacoes.append(parcela / total if total > 0 else float("nan"))
        # Somar o que ficou fora do Top 10% no ranking deste indicador.
        restante = valores.iloc[quantidade:].sum()
        participacoes.append(restante / total if total > 0 else float("nan"))
        resumo[indicador] = participacoes

    primeiro_ano, ultimo_ano = periodos_g3.loc[cultura]
    with plt.ioff():
        ax = (resumo * 100).plot.bar(
            ylabel="Participação no total nacional (%)",
            color=CORES_CREDITO_AGRICULTURA,
            figsize=(11, 6),
            rot=0,
            width=0.7,
            ylim=(0, 100),
        )
        ax.set_title(
            f"Concentração por município · {cultura} ({primeiro_ano} a {ultimo_ano})", pad=32,
        )
        ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False)
        ax.get_figure().tight_layout()
        buffer = BytesIO()
        ax.get_figure().savefig(buffer, format="png", dpi=100, bbox_inches="tight")
        imagem = buffer.getvalue()
        buffer.close()
        plt.close(ax.get_figure())

    return imagem


def exibir_grafico_3():
    seletor_g3 = Dropdown(
        options=periodos_g3.index.tolist(), description="Cultura", layout=Layout(width="330px"),
    )
    imagem_g3 = ImagemWidget(format="png", layout=Layout(max_width="100%"))


    def atualizar_imagem_g3(mudanca=None):
        imagem_g3.value = grafico_3(seletor_g3.value) or b""


    seletor_g3.observe(atualizar_imagem_g3, names="value")
    atualizar_imagem_g3()
    display(VBox([seletor_g3, imagem_g3]))


### Gráfico 4 — Crédito entre municípios com áreas semelhantes

Apoia a pergunta 4: cada ponto representa um município, com área plantada no eixo horizontal, crédito nominal no vertical e cor por região. Use os seletores de cultura, ano e UF, o zoom e os detalhes ao passar o mouse; diferenças verticais entre pontos com áreas próximas revelam variação do crédito mesmo entre municípios de escala semelhante. A comparação usa apenas a cobertura validada, não estabelece causas e ainda não contempla contratos municipais, cuja contagem está ausente.


In [ ]:
def criar_grafico_dispersao_area_credito(tabela, cultura="soja", ano=None, uf="Todas"):
    """Compara área e crédito dos municípios no recorte selecionado."""
    if ano is None:
        ano = int(tabela["ano"].max())

    filtro = (tabela["cultura"] == cultura) & (tabela["ano"] == ano)
    dados = tabela.loc[filtro]

    if uf != "Todas":
        dados = dados.loc[dados["uf"] == uf]

    grafico = alt.Chart(dados).mark_circle(size=45, opacity=0.5, clip=True)
    grafico = grafico.encode(
        x=alt.X("area_plantada_ha:Q", title="Área plantada (ha)", axis=alt.Axis(format=",.0f")),
        y=alt.Y("credito_nominal_brl:Q", title="Crédito de custeio (R$ nominais)", axis=alt.Axis(format=",.0f")),
        color=alt.Color("regiao:N", scale=CORES_REGIOES, sort=ORDEM_REGIOES, title="Região"),
        tooltip=[
            alt.Tooltip("municipio:N", title="Município"),
            alt.Tooltip("codigo_municipio:N", title="Código IBGE"),
            alt.Tooltip("uf:N", title="UF"),
            alt.Tooltip("cultura:N", title="Cultura"),
            alt.Tooltip("ano:O", title="Ano"),
            alt.Tooltip("area_plantada_ha:Q", title="Área (ha)", format=",.2f"),
            alt.Tooltip("credito_nominal_brl:Q", title="Crédito (R$)", format=",.2f"),
            alt.Tooltip("faixa_hectares:N", title="Faixa de área"),
        ],
    )

    return grafico.properties(
        width=850,
        height=450,
        title=f"Crédito e área plantada — {cultura} ({ano})",
    ).interactive()


def atualizar_dispersao_area_credito(cultura, ano, uf):
    grafico = criar_grafico_dispersao_area_credito(tabela_categorizada, cultura, ano, uf)
    with alt.renderers.enable("default", embed_options={"actions": False}):
        display(grafico)


def exibir_grafico_4():
    culturas_q4 = sorted(tabela_categorizada["cultura"].unique().tolist())
    anos_q4 = sorted(tabela_categorizada["ano"].astype(int).unique().tolist())
    ufs_q4 = ["Todas"] + sorted(tabela_categorizada["uf"].unique().tolist())

    seletor_cultura_q4 = Dropdown(
        options=culturas_q4, value="soja", description="Cultura", layout=Layout(width="330px"),
    )
    seletor_ano_q4 = Dropdown(options=anos_q4, value=anos_q4[-1], description="Ano")
    seletor_uf_q4 = Dropdown(options=ufs_q4, value="Todas", description="UF")

    grafico_inicial = criar_grafico_dispersao_area_credito(
        tabela_categorizada, seletor_cultura_q4.value, seletor_ano_q4.value, seletor_uf_q4.value,
    )
    saida_q4 = alt.JupyterChart(grafico_inicial, embed_options={"actions": False})

    def atualizar_saida_q4(mudanca=None):
        saida_q4.chart = criar_grafico_dispersao_area_credito(
            tabela_categorizada, seletor_cultura_q4.value, seletor_ano_q4.value, seletor_uf_q4.value,
        )

    for seletor in (seletor_cultura_q4, seletor_ano_q4, seletor_uf_q4):
        seletor.observe(atualizar_saida_q4, names="value")

    controles_q4 = HBox([seletor_cultura_q4, seletor_ano_q4, seletor_uf_q4])
    display(VBox([controles_q4, saida_q4]))


### Gráfico 5 — Crédito por tonelada e por hectare entre estados

Responde à parte estadual da pergunta 5 com a soma do crédito dividida pela soma da produção ou da área dos mesmos municípios, por cultura e ano. Barras maiores indicam maior intensidade de crédito e a linha tracejada mostra a referência nacional da mesma cobertura; os painéis têm unidades e ordenações próprias. Os seletores atualizam ambos os gráficos, e os detalhes mostram totais e municípios utilizados; produção inválida ou não positiva é excluída apenas de R$/t, e os resultados não representam necessariamente a totalidade de cada UF. O menu de opções do Altair fica oculto; seletores e detalhes permanecem disponíveis.


In [101]:
def calcular_indicadores_estaduais(tabela):
    """Calcula crédito por produção ou área dos mesmos municípios de cada UF."""
    partes = []
    indicadores = [
        ("credito_por_tonelada", "producao_t", "R$/t"),
        ("credito_por_hectare", "area_plantada_ha", "R$/ha"),
    ]

    for indicador, coluna, unidade in indicadores:
        denominador = pd.to_numeric(tabela[coluna], errors="coerce")
        valido = denominador.gt(0) & denominador.lt(float("inf"))
        base = tabela.loc[valido.fillna(False)].copy()
        base[coluna] = denominador.loc[base.index]

        excluidos = len(tabela) - len(base)
        print(f"{unidade}: {excluidos:,} registros excluídos por denominador inválido ou não positivo.")

        resumo = base.groupby(["cultura", "ano", "uf"], as_index=False).agg(
            credito_total_brl=("credito_nominal_brl", "sum"),
            denominador_total=(coluna, "sum"),
            numero_municipios=("codigo_municipio", "nunique"),
        )
        resumo["indicador"] = indicador
        resumo["unidade"] = unidade
        resumo["valor"] = resumo["credito_total_brl"] / resumo["denominador_total"]

        totais = ["credito_total_brl", "denominador_total"]
        nacional = resumo.groupby(["cultura", "ano"])[totais].transform("sum")
        resumo["referencia_nacional"] = nacional["credito_total_brl"] / nacional["denominador_total"]
        partes.append(resumo)

    return pd.concat(partes, ignore_index=True)


tabela_indicadores_estaduais = calcular_indicadores_estaduais(tabela_validada)
tabela_indicadores_estaduais.head()


R$/t: 46 registros excluídos por denominador inválido ou não positivo.
R$/ha: 0 registros excluídos por denominador inválido ou não positivo.


,cultura,ano,uf,credito_total_brl,denominador_total,numero_municipios,indicador,unidade,valor,referencia_nacional
0,cana_de_acucar,2016,AL,3.873530e+07,17790075.0,39,credito_por_tonelada,R$/t,2.177355,5.596106
1,cana_de_acucar,2016,BA,5.437237e+07,4270212.0,8,credito_por_tonelada,R$/t,12.732945,5.596106
2,cana_de_acucar,2016,CE,3.457100e+04,96800.0,2,credito_por_tonelada,R$/t,0.357138,5.596106
3,cana_de_acucar,2016,ES,1.882088e+06,1308232.0,10,credito_por_tonelada,R$/t,1.438650,5.596106
4,cana_de_acucar,2016,GO,2.554035e+08,57056438.0,37,credito_por_tonelada,R$/t,4.476331,5.596106


In [ ]:
def criar_graficos_estaduais_questao5(tabela, cultura="soja", ano=None):
    """Compara crédito por tonelada e por hectare nos estados cobertos."""
    if ano is None:
        ano = int(tabela["ano"].max())

    filtro = (tabela["cultura"] == cultura) & (tabela["ano"] == ano)
    recorte = tabela.loc[filtro]
    paineis = []
    indicadores = [
        ("tonelada", "R$/t", "Produção coberta (t)", AZUL),
        ("hectare", "R$/ha", "Área coberta (ha)", VERDE),
    ]

    for medida, unidade, denominador, cor in indicadores:
        dados = recorte.loc[recorte["indicador"] == f"credito_por_{medida}"]
        barras = alt.Chart(dados).mark_bar(color=cor).encode(
            y=alt.Y("uf:N", sort="-x", title="UF"),
            x=alt.X("valor:Q", title=f"Crédito ({unidade})", axis=alt.Axis(format=",.0f")),
            tooltip=[
                alt.Tooltip("uf:N", title="UF"),
                alt.Tooltip("cultura:N", title="Cultura"),
                alt.Tooltip("ano:O", title="Ano"),
                alt.Tooltip("valor:Q", title=f"Crédito ({unidade})", format=",.2f"),
                alt.Tooltip("credito_total_brl:Q", title="Crédito coberto (R$)", format=",.2f"),
                alt.Tooltip("denominador_total:Q", title=denominador, format=",.2f"),
                alt.Tooltip("numero_municipios:Q", title="Municípios cobertos", format=",.0f"),
                alt.Tooltip("referencia_nacional:Q", title="Referência nacional", format=",.2f"),
            ],
        )

        referencia = alt.Chart(dados.head(1)).mark_rule(
            color=VERMELHO, strokeDash=[5, 4], strokeWidth=2,
        ).encode(
            x="referencia_nacional:Q",
            tooltip=alt.Tooltip("referencia_nacional:Q", title="Referência nacional", format=",.2f"),
        )

        painel = (barras + referencia).properties(
            width=390, height=460, title=f"Crédito por {medida}",
        )
        paineis.append(painel)

    return alt.hconcat(*paineis, spacing=50).resolve_scale(
        y="independent", x="independent",
    ).properties(title=alt.Title(
        f"Questão 5 — Crédito entre estados — {cultura} ({ano})",
        subtitle="Linha tracejada: referência nacional dos municípios cobertos; valores nominais.",
    ))


def atualizar_graficos_estaduais_questao5(cultura, ano):
    grafico = criar_graficos_estaduais_questao5(tabela_indicadores_estaduais, cultura, ano)
    with alt.renderers.enable("default", embed_options={"actions": False}):
        display(grafico)


def exibir_grafico_5():
    culturas_q5 = sorted(tabela_indicadores_estaduais["cultura"].unique().tolist())
    anos_q5 = sorted(tabela_indicadores_estaduais["ano"].astype(int).unique().tolist())

    seletor_cultura_q5 = Dropdown(
        options=culturas_q5, value="soja", description="Cultura", layout=Layout(width="330px"),
    )
    seletor_ano_q5 = Dropdown(options=anos_q5, value=anos_q5[-1], description="Ano")

    grafico_inicial = criar_graficos_estaduais_questao5(
        tabela_indicadores_estaduais, seletor_cultura_q5.value, seletor_ano_q5.value,
    )
    saida_q5 = alt.JupyterChart(grafico_inicial, embed_options={"actions": False})

    def atualizar_saida_q5(mudanca=None):
        saida_q5.chart = criar_graficos_estaduais_questao5(
            tabela_indicadores_estaduais, seletor_cultura_q5.value, seletor_ano_q5.value,
        )

    for seletor in (seletor_cultura_q5, seletor_ano_q5):
        seletor.observe(atualizar_saida_q5, names="value")

    controles_q5 = HBox([seletor_cultura_q5, seletor_ano_q5])
    display(VBox([controles_q5, saida_q5]))


## Malhas dos gráficos 6 e 7

As malhas municipais e estaduais de 2020 são lidas com GeoPandas a partir do GeoJSON simplificado da [API oficial de malhas do IBGE](https://servicodados.ibge.gov.br/api/docs/malhas?versao=3). O cache em `dados/ibge/malhas` registra URL e data da obtenção e evita novos downloads; a primeira obtenção exige rede e mantém a validação HTTPS. Os códigos IBGE ligam os polígonos aos dados, sem geobr, DuckDB ou alterações em bibliotecas; GeoPandas e NumPy ficam restritos à preparação destas seções. Registros sem polígono correspondente são informados e continuam nas estatísticas, mas não são desenhados no mapa.


In [ ]:
ANO_MALHA = 2020
CACHE_MALHAS = Path("dados/ibge/malhas")
SIGLAS_UF = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE", "29": "BA",
    "31": "MG", "32": "ES", "33": "RJ", "35": "SP", "41": "PR", "42": "SC", "43": "RS",
    "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}


def carregar_malha_ibge(nivel):
    """Lê GeoJSON do cache ou obtém a malha oficial com validação HTTPS."""
    arquivo = CACHE_MALHAS / f"{nivel.lower()}_{ANO_MALHA}.json"

    if arquivo.exists():
        registro = json.loads(arquivo.read_text(encoding="utf-8"))
    else:
        url = "https://servicodados.ibge.gov.br/api/v3/malhas/paises/BR"
        parametros_malha = {
            "periodo": ANO_MALHA, "intrarregiao": nivel,
            "qualidade": "minima", "formato": "application/vnd.geo+json",
        }
        with rqs.Session() as sessao:
            certificado = Path("certs/SectigoPublicServerAuthenticationCADVR36.pem")
            sessao.mount("https://servicodados.ibge.gov.br/", IBGETLSAdapter(certificado))
            resposta = sessao.get(url, params=parametros_malha, timeout=60)
            resposta.raise_for_status()
            geojson = resposta.json()

        if not geojson.get("features"):
            raise ValueError("O IBGE não retornou uma malha válida.")

        registro = {
            "url": resposta.url,
            "obtido_em_utc": datetime.now(timezone.utc).isoformat(),
            "dados": geojson,
        }
        CACHE_MALHAS.mkdir(parents=True, exist_ok=True)
        temporario = arquivo.with_suffix(".tmp")
        temporario.write_text(json.dumps(registro, ensure_ascii=False), encoding="utf-8")
        temporario.replace(arquivo)

    mapa = gpd.GeoDataFrame.from_features(registro["dados"]["features"], crs="EPSG:4326")
    mapa["codarea"] = mapa["codarea"].astype(str)
    mapa["abbrev_state"] = mapa["codarea"].str[:2].map(SIGLAS_UF)
    return mapa


mapa_municipios = carregar_malha_ibge("municipio").rename(columns={"codarea": "municipio_id"})
mapa_estados = carregar_malha_ibge("UF")
mapa_municipios["regiao"] = mapa_municipios["municipio_id"].str[0].map(REGIOES)

recortes_g67 = {"Brasil": sorted(mapa_municipios["abbrev_state"].unique().tolist())}
for nome in REGIOES.values():
    estados = mapa_municipios.loc[mapa_municipios["regiao"] == nome, "abbrev_state"]
    recortes_g67[nome] = sorted(estados.unique().tolist())
for uf in recortes_g67["Brasil"]:
    recortes_g67[uf] = [uf]


def desenhar_fundo(eixo, ufs):
    municipios = mapa_municipios.loc[mapa_municipios["abbrev_state"].isin(ufs)]
    estados = mapa_estados.loc[mapa_estados["abbrev_state"].isin(ufs)]
    municipios.plot(ax=eixo, color=CINZA_FUNDO, linewidth=0)
    estados.boundary.plot(ax=eixo, color=CINZA_CONTORNO, linewidth=0.45)
    eixo.set_axis_off()


### Dados compartilhados dos gráficos 6 e 7

Prepara uma única base municipal de crédito, área e produtividade para os dois gráficos, mantendo os filtros de área mínima de 50 ha, crédito positivo e rendimento positivo. As listas de culturas e anos são obtidas dessa base e usadas nos dois seletores.


In [ ]:
# Mantém a mesma preparação: crédito anual e indicadores agrícolas por código IBGE.
credito_g67 = sicor.assign(
    ano=sicor["ano"].astype(int), municipio_id=sicor["codIbge"].astype(str),
)
credito_g67 = credito_g67.groupby(
    ["cultura", "ano", "municipio_id"], as_index=False,
)["valor de custeio"].sum().rename(columns={"valor de custeio": "credito"})

ibge_g67 = tabela_ibge_municipal.assign(
    municipio_id=tabela_ibge_municipal["codigo municipio"].astype(str),
).rename(columns={"area plantada (hectares)": "area"})

base_g67 = ibge_g67.merge(credito_g67, on=["cultura", "ano", "municipio_id"])
filtro = (
    (base_g67["area"] >= AREA_MINIMA_HA)
    & (base_g67["credito"] > 0)
    & (base_g67["rendimento medio (kg por hectare)"] > 0)
)
base_g67 = base_g67.loc[filtro].copy()
base_g67["credito_ha"] = base_g67["credito"] / base_g67["area"]
base_g67["produtividade"] = base_g67["rendimento medio (kg por hectare)"] / 1000

sem_malha = ~base_g67["municipio_id"].isin(mapa_municipios["municipio_id"])
if sem_malha.any():
    print(f"{sem_malha.sum()} registros sem polígono de 2020: mantidos nas estatísticas, ausentes dos mapas.")

culturas_g67 = sorted(base_g67["cultura"].unique().tolist())
anos_g67 = sorted(base_g67["ano"].astype(int).unique().tolist())


### Gráfico 6 — Crédito por hectare e produtividade

Responde à pergunta 6 comparando, na mesma cultura, ano e recorte, os quartis municipais de crédito por hectare e de produtividade nos mapas, e a distribuição da produtividade por faixa de crédito no boxplot. Compare as medianas e a dispersão das caixas; coincidências espaciais sugerem associação, não causalidade. Mantém municípios com pelo menos 50 ha, crédito e rendimento positivos; empates nos quartis são desempatados pela ordem dos registros e os pontos extremos não são exibidos no boxplot. Cinza indica ausência ou exclusão; os seletores alternam cultura, ano e recorte.


In [ ]:
faixas_g6 = ["Q1", "Q2", "Q3", "Q4"]


def desenhar_quartis(eixo, dados, coluna, cores, titulo, ufs):
    desenhar_fundo(eixo, ufs)
    dados.plot(ax=eixo, column=coluna, categorical=True, cmap=ListedColormap(cores), linewidth=0)

    legenda = []
    for faixa, cor in zip(faixas_g6, cores):
        legenda.append(Patch(facecolor=cor, label=faixa))
    legenda.append(Patch(facecolor=CINZA_FUNDO, label="Sem dados / excluído"))

    eixo.legend(handles=legenda, loc="lower center", bbox_to_anchor=(0.5, -0.17),
                ncol=3, frameon=False, fontsize=8)
    eixo.set_title(titulo, fontsize=12, pad=12)


def grafico_6(cultura, ano, recorte):
    ufs = recortes_g67[recorte]
    filtro = (base_g67["cultura"] == cultura) & (base_g67["ano"] == ano) & base_g67["estado"].isin(ufs)
    dados = base_g67.loc[filtro].copy()

    if len(dados) < 8:
        print("Poucos municípios para comparar os quartis neste recorte.")
        return

    dados["faixa_credito"] = pd.qcut(dados["credito_ha"].rank(method="first"), 4, labels=faixas_g6)
    dados["faixa_produtividade"] = pd.qcut(dados["produtividade"].rank(method="first"), 4, labels=faixas_g6)
    dados_mapa = mapa_municipios[["municipio_id", "geometry"]].merge(dados, on="municipio_id")

    with plt.ioff():
        fig, eixos = plt.subplots(1, 3, figsize=(15, 5.8), layout="constrained")
        desenhar_quartis(eixos[0], dados_mapa, "faixa_credito", TONS_AZUL, "Crédito por hectare", ufs)
        desenhar_quartis(eixos[1], dados_mapa, "faixa_produtividade", TONS_VERDE, "Produtividade (t/ha)", ufs)

        grupos = []
        rotulos = []
        for faixa in faixas_g6:
            grupo = dados.loc[dados["faixa_credito"] == faixa, "produtividade"]
            grupos.append(grupo)
            rotulos.append(f"{faixa}\nn = {len(grupo)}")

        eixo = eixos[2]
        caixas = eixo.boxplot(grupos, patch_artist=True, showfliers=False, widths=0.55,
                             medianprops={"color": CINZA_TEXTO, "linewidth": 2})
        for caixa, cor in zip(caixas["boxes"], TONS_AZUL):
            caixa.set(facecolor=cor, edgecolor=CINZA_CONTORNO)
        for posicao, grupo in enumerate(grupos, start=1):
            eixo.annotate(f"{grupo.median():.1f}", (posicao, grupo.median()),
                          xytext=(7, 5), textcoords="offset points", fontsize=9)

        eixo.set_xticks(range(1, 5), labels=rotulos)
        eixo.set(xlabel="Quartil de crédito por hectare", ylabel="Produtividade (t/ha)")
        eixo.set_title("Produtividade por faixa de crédito", fontsize=12, pad=12)
        eixo.set_axisbelow(True)
        eixo.grid(axis="y", alpha=0.2)
        eixo.spines[["top", "right"]].set_visible(False)
        fig.suptitle(f"Crédito e produtividade — {cultura} · {ano} · {recorte}", fontsize=15)
        buffer = BytesIO()
        fig.savefig(buffer, format="png", dpi=100, bbox_inches="tight")
        imagem = buffer.getvalue()
        buffer.close()
        plt.close(fig)

    return imagem


def exibir_grafico_6():
    seletor_cultura_g6 = Dropdown(options=culturas_g67, description="Cultura", layout=Layout(width="330px"))
    seletor_ano_g6 = Dropdown(options=anos_g67, value=anos_g67[-1], description="Ano")
    seletor_recorte_g6 = Dropdown(options=list(recortes_g67), description="Recorte")
    imagem_g6 = ImagemWidget(format="png", layout=Layout(max_width="100%"))


    def atualizar_imagem_g6(mudanca=None):
        imagem_g6.value = grafico_6(
            seletor_cultura_g6.value, seletor_ano_g6.value, seletor_recorte_g6.value,
        ) or b""


    for seletor in (seletor_cultura_g6, seletor_ano_g6, seletor_recorte_g6):
        seletor.observe(atualizar_imagem_g6, names="value")

    atualizar_imagem_g6()
    controles_g6 = HBox([seletor_cultura_g6, seletor_ano_g6, seletor_recorte_g6])
    display(VBox([controles_g6, imagem_g6]))


### Gráfico 7 — Municípios atípicos no padrão estadual

Responde à pergunta 7 destacando valores fora de Q1 − 1,5 × IQR e Q3 + 1,5 × IQR na própria UF: azul para crédito por hectare, verde para produtividade e vermelho para ambos. O crédito é avaliado em log10 e UFs com menos de dez municípios elegíveis ficam fora; o recorte altera a exibição, não o padrão estadual. Mantém o mínimo de 50 ha e a base do gráfico 6; o mapa nomeia até seis casos ao selecionar uma UF, sem tabela complementar. Atípico não significa erro ou efeito causal.


In [ ]:
def sinalizar_atipicos(dados):
    """Compara cada município aos quartis e à mediana da própria UF."""
    dados = dados.copy()
    dados["log_credito_ha"] = np.log10(dados["credito_ha"])

    for coluna, nome in [("log_credito_ha", "credito"), ("produtividade", "produtividade")]:
        grupo = dados.groupby("estado")[coluna]
        quartis = grupo.quantile([0.25, 0.75]).unstack()
        q1 = dados["estado"].map(quartis[0.25])
        q3 = dados["estado"].map(quartis[0.75])
        iqr = q3 - q1
        mediana = grupo.transform("median")

        fora = (dados[coluna] < q1 - 1.5 * iqr) | (dados[coluna] > q3 + 1.5 * iqr)
        dados[f"atipico_{nome}"] = fora & (iqr > 0)
        escala = (iqr / 1.349).replace(0, float("nan"))
        dados[f"escore_{nome}"] = ((dados[coluna] - mediana) / escala).abs()

    dados["tipo"] = "Típico"
    dados.loc[dados["atipico_credito"], "tipo"] = "Crédito/ha atípico"
    dados.loc[dados["atipico_produtividade"], "tipo"] = "Produtividade atípica"
    ambos = dados["atipico_credito"] & dados["atipico_produtividade"]
    dados.loc[ambos, "tipo"] = "Ambos"
    dados["escore"] = dados[["escore_credito", "escore_produtividade"]].max(axis=1)
    return dados


def grafico_7(cultura, ano, recorte):
    ufs = recortes_g67[recorte]
    dados = base_g67.loc[(base_g67["cultura"] == cultura) & (base_g67["ano"] == ano)]
    contagem = dados["estado"].value_counts()
    dados = dados.loc[dados["estado"].map(contagem) >= MIN_MUNICIPIOS_UF]

    if dados.empty:
        print("Nenhuma UF tem municípios suficientes para esta cultura e ano.")
        return

    # Calcula o padrão na UF inteira antes de filtrar o recorte exibido.
    dados = sinalizar_atipicos(dados)
    dados = dados.loc[dados["estado"].isin(ufs)]
    if dados.empty:
        print("Nenhum município elegível no recorte selecionado.")
        return

    dados_mapa = mapa_municipios[["municipio_id", "geometry"]].merge(dados, on="municipio_id")
    with plt.ioff():
        fig, eixo = plt.subplots(figsize=(11, 7.5), layout="constrained")
        desenhar_fundo(eixo, ufs)
        tipicos = dados_mapa.loc[dados_mapa["tipo"] == "Típico"]
        if not tipicos.empty:
            tipicos.plot(ax=eixo, color=CINZA_TIPICOS, linewidth=0)

        atipicos = dados_mapa.loc[dados_mapa["tipo"] != "Típico"].copy()
        centros = atipicos.geometry.representative_point()
        tamanho = 22
        if len(ufs) <= 9:
            tamanho = 45
        if len(ufs) == 1:
            tamanho = 110

        legenda = [Patch(facecolor=CINZA_TIPICOS, label="Típico"),
                   Patch(facecolor=CINZA_FUNDO, label="Sem dados / excluído")]
        for tipo, cor in CORES_ATIPICOS.items():
            parte = atipicos.loc[atipicos["tipo"] == tipo]
            if not parte.empty:
                parte.plot(ax=eixo, color=cor, linewidth=0)
                pontos = centros.loc[parte.index]
                eixo.scatter(pontos.x, pontos.y, s=tamanho, color=cor,
                             edgecolor="white", linewidth=0.4, zorder=3)
            legenda.append(Patch(facecolor=cor, label=f"{tipo} ({len(parte)})"))

        if len(ufs) == 1:
            for indice, linha in atipicos.nlargest(6, "escore").iterrows():
                ponto = centros.loc[indice]
                eixo.annotate(linha["municipio"], (ponto.x, ponto.y),
                              xytext=(6, 6), textcoords="offset points", fontsize=8)

        eixo.legend(handles=legenda, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=9)
        eixo.set_title(f"Municípios atípicos — {cultura} · {ano} · {recorte}", fontsize=15, pad=15)
        buffer = BytesIO()
        fig.savefig(buffer, format="png", dpi=100, bbox_inches="tight")
        imagem = buffer.getvalue()
        buffer.close()
        plt.close(fig)

    return imagem


def exibir_grafico_7():
    seletor_cultura_g7 = Dropdown(options=culturas_g67, description="Cultura", layout=Layout(width="330px"))
    seletor_ano_g7 = Dropdown(options=anos_g67, value=anos_g67[-1], description="Ano")
    seletor_recorte_g7 = Dropdown(options=list(recortes_g67), description="Recorte")
    imagem_g7 = ImagemWidget(format="png", layout=Layout(max_width="100%"))


    def atualizar_imagem_g7(mudanca=None):
        imagem_g7.value = grafico_7(
            seletor_cultura_g7.value, seletor_ano_g7.value, seletor_recorte_g7.value,
        ) or b""


    for seletor in (seletor_cultura_g7, seletor_ano_g7, seletor_recorte_g7):
        seletor.observe(atualizar_imagem_g7, names="value")

    atualizar_imagem_g7()
    controles_g7 = HBox([seletor_cultura_g7, seletor_ano_g7, seletor_recorte_g7])
    display(VBox([controles_g7, imagem_g7]))
